In [1]:
import torch
import esm
from Bio.Seq import Seq
from Bio import SeqIO
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from mpl_toolkits.mplot3d import Axes3D
seq_file='/media/ktht/ExtraUbuntu/amromics/panta2/panta/out/test/g_mmseq_a_ems_c_mcl_unique/temp/combined.faa'

In [2]:
def protein_sequences_to_vector_ems(fasta_file):
    starttime = datetime.now()
    index_seq_id=[]
    list_vectors=[]
    data=[]
    model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    batch_converter = alphabet.get_batch_converter()
    model.eval()
    
    with open(fasta_file) as handle:
        for record in SeqIO.parse(handle, "fasta"):          
            index_seq_id.append(record.id)          
            data.append((record.id,str(record.seq)))
    batch_labels, batch_strs, batch_tokens = batch_converter(data)
    elapsed = datetime.now() - starttime
    
    for i in range(len(batch_tokens)):
        list_vectors.append(batch_tokens[i].numpy().tolist())
    print(f'protein to {len(list_vectors)} vector by ems -- time taken {elapsed}')
    return batch_labels,list_vectors

In [3]:
index_seq,vectors=protein_sequences_to_vector_ems(seq_file)

        

protein to 63725 vector by ems -- time taken 0:02:28.439041


In [ ]:
X=np.array(vectors)
starttime = datetime.now()
hierarchical = AgglomerativeClustering(n_clusters=None,distance_threshold=150,compute_full_tree=True)
labels_hierarchical = hierarchical.fit_predict(X)
elapsed = datetime.now() - starttime
print(f'clustering -- time taken {elapsed}')
clusters={}

for i in range(len(labels_hierarchical)):
    if labels_hierarchical[i] not in clusters.keys():
        clusters[labels_hierarchical[i]]=[]
    clusters[labels_hierarchical[i]].append(index_seq[i])
print(len(clusters))